<a href="https://colab.research.google.com/github/Juana-Zhang/Casual-Inference/blob/main/Project_Evaluating_Marketing_Impact_via_Propensity_Score_Matching_(PSM).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Project Overview
This project demonstrates how to estimate the **true incremental effect** of a marketing intervention (e.g., a discount voucher) using observational data. In real-world tech scenarios, "treatment" is rarely random; high-value or highly active users are often more likely to receive incentives, leading to **Selection Bias**. This project uses **Propensity Score Matching (PSM)** to de-bias the data and recover the actual causal lift.

## 2. Synthetic Data Generation (The Logic)
To simulate a realistic tech environment, we generated 2,000 synthetic user records with built-in **Selection Bias**:

* **Confounders ($X$)**: Features that influence both treatment and outcome.
    * `app_freq`: User's weekly app opening frequency.
    * `hist_gmv`: Historical spending levels.
    * `is_member`: Binary status of loyalty membership.
* **Treatment Assignment ($T$)**: We used a **Logistic Function** to ensure high-value users (higher $X$) were more likely to receive the voucher.
    * $P(T=1|X) = \sigma(-5 + 0.3 \cdot \text{freq} + 0.002 \cdot \text{gmv} + 1.5 \cdot \text{member})$.
* **Outcome Generation ($Y$)**: We set the **True ATE (Average Treatment Effect)** to exactly **30 units**.
    * $Y = 100 + 30 \cdot T + \text{Confounders} + \epsilon$.

In [2]:
# Install the library first
!pip install -q causalinference
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from causalinference import CausalModel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 2.1 MB/s eta 0:00:00


In [3]:
# 1. Simulate Internet Ads Data
# Scenario: Evaluating the impact of a voucher on user GMV
np.random.seed(42)
n = 2000
# Features (Confounders):
# App usage frequency, historical spending, and loyalty membership status
app_frequency = np.random.normal(10, 3, n)
history_gmv = np.random.normal(500, 100, n)
is_member = np.random.binomial(1, 0.4, n) #Bernoulli distribution, p =0.4


In [4]:
# 2. Generate Treatment Assignment (Selection Bias)
# Users with higher frequency and membership are more likely to receive vouchers
logit_ps = -5 + 0.3 * app_frequency + 0.002 * history_gmv + 1.5 * is_member
prob_t = 1 / (1 + np.exp(-logit_ps))
T = np.random.binomial(1, prob_t)

In [5]:
# 3. Generate Outcome (GMV)
# Assume the true incremental effect (ATE) is 30 units
true_lift = 30
Y = 100 + true_lift * T + 0.5 * history_gmv + 2 * app_frequency   + np.random.normal(0, 10, n)

df = pd.DataFrame({
    'app_freq': app_frequency,
    'hist_gmv': history_gmv,
    'member': is_member,
    'treatment': T,
    'gmv': Y
})
df.head()

,app_freq,hist_gmv,member,treatment,gmv
0,11.490142,432.482173,0,0,345.117763
1,9.585207,485.548133,0,0,335.426200
2,11.943066,420.758008,1,1,377.929029
3,14.569090,469.203847,0,1,386.362736
4,9.297540,310.638533,1,1,310.780024


## 3. Methodology: Recovering the Causal Effect

### * Step 1: Propensity Score Estimation
Using `causalinference`, we run a Logistic Regression to "reverse-engineer" the treatment assignment probability ($P$) for each user.

### * Step 2: Nearest Neighbor Matching
For every treated user, we match them with a "control twin" who has a similar Propensity Score but did not receive the voucher.

### * Step 3: Covariate Balance Check
We compare the **Standardized Mean Difference (Nor-diff)** before and after matching.
* **Success Criteria**: Achieving a $Nor-diff < 0.1$ across all features, effectively simulating a randomized experiment.

In [6]:
# 4. Execute Propensity Score Matching (PSM)
X = df[['app_freq', 'hist_gmv', 'member']].values
causal = CausalModel(df['gmv'].values, df['treatment'].values, X)

# Estimate Propensity Scores (T ~ X) using Logistic Regression
causal.est_propensity()

# Perform Matching to estimate Average Treatment Effect (ATE)
causal.est_via_matching(bias_adj=True)

I used bias_adj=True to perform a linear regression adjustment on these matched sets. This removes the large-sample bias that occurs when the match isn't perfect, ensuring my ATE estimate is as pure and accurate as possible."

## 4. Key Results

| Metric | Value | Interpretation |
| :--- | :--- | :--- |
| **Naive Comparison** | 43.24 | Biased; includes pre-existing user differences |
| **PSM Corrected ATE** | **29.69** | **True Causal Effect**; matches the simulated ground truth |
| **P-Value** | 0.000 | The results are statistically significant |

In [7]:
# 5. Output Results
print(f"Naive Comparison (Bias): {df[df.treatment==1].gmv.mean() - df[df.treatment==0].gmv.mean():.2f}")
print("\n--- PSM Corrected ATE ---")
print(causal.estimates)

Naive Comparison (Bias): 43.24

--- PSM Corrected ATE ---

Treatment Effect Estimates: Matching

                     Est.       S.e.          z      P>|z|      [95% Conf. int.]
--------------------------------------------------------------------------------
           ATE     29.686      0.896     33.142      0.000     27.930     31.441
           ATC     29.805      1.050     28.386      0.000     27.747     31.863
           ATT     29.514      1.091     27.049      0.000     27.375     31.652



In [8]:
# 6. Balance Check (Crucial for Tech Interviews)
print("\n--- Covariate Balance Check ---")
print(causal.summary_stats)


--- Covariate Balance Check ---

Summary Statistics

                      Controls (N_c=1178)         Treated (N_t=822)             
       Variable         Mean         S.d.         Mean         S.d.     Raw-diff
--------------------------------------------------------------------------------
              Y      364.756       50.841      407.992       51.741       43.237

                      Controls (N_c=1178)         Treated (N_t=822)             
       Variable         Mean         S.d.         Mean         S.d.     Nor-diff
--------------------------------------------------------------------------------
             X0        9.267        2.733       11.379        2.842        0.758
             X1      491.981       99.807      509.925      100.692        0.179
             X2        0.276        0.447        0.538        0.499        0.553



## 5. Conclusion
By simulating the data ourselves, we proved that PSM can effectively strip away **Selection Bias**. This methodology allows data scientists to provide the business with a clean **Incremental ROI**, preventing wasted marketing spend on users who would have purchased anyway.

In [9]:
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

# 1. Get Propensity Scores
# These were previously estimated using causal.est_propensity()
df['pscore'] = causal.propensity['fitted']

# 2. Split data into treatment and control groups
treated = df[df['treatment'] == 1]
control = df[df['treatment'] == 0]

# 3. Find the best matched control "counterparts" for each treated user
# Implement Nearest Neighbor Matching
knn = NearestNeighbors(n_neighbors=1, metric='manhattan')
knn.fit(control[['pscore']])
distances, indices = knn.kneighbors(treated[['pscore']])

# Extract the matched control samples
matched_control = control.iloc[indices.flatten()]

# 4. Calculate post-matching Standardized Mean Difference (SMD / Nor-diff)
# This measures how well the covariates are balanced between groups
def calc_smd(group1, group2, feature):
    m1, m2 = group1[feature].mean(), group2[feature].mean()
    v1, v2 = group1[feature].var(), group2[feature].var()
    return (m1 - m2) / np.sqrt((v1 + v2) / 2)

# Define features and extract SMD before and after matching
features = ['app_freq', 'hist_gmv', 'member']
# Extract raw Nor-diff from the summary_stats attribute
raw_smds = [causal.summary_stats['ndiff'][i] for i in range(len(features))]
matched_smds = [calc_smd(treated, matched_control, f) for f in features]

# 5. Print comparison results
# This table proves whether the selection bias was successfully removed
balance_df = pd.DataFrame({
    'Feature': features,
    'Before (Raw Nor-diff)': raw_smds,
    'After (Matched Nor-diff)': matched_smds
})
print("--- Covariate Balance Comparison ---")
print(balance_df)

--- Covariate Balance Comparison ---
    Feature  Before (Raw Nor-diff)  After (Matched Nor-diff)
0  app_freq               0.757667                  0.049881
1  hist_gmv               0.178992                  0.026219
2    member               0.552690                 -0.068537
